# Q26: Types of Optimizers – Adam, Adagrad, etc.
**Topic:** Classical-ml | **Difficulty:** Medium | **Time:** 15 min
**Sheet:** Grind75ML

---

## Question
Explain the types of optimizers – Adam, Adagrad, etc.

## Detailed Answer

### 1. SGD (Stochastic Gradient Descent)
$$\theta_{t+1} = \theta_t - \alpha \cdot g_t$$
- Simple, no adaptive learning rate
- Often best final performance with proper tuning
- Needs careful LR scheduling

### 2. SGD with Momentum
$$v_t = \beta v_{t-1} + g_t \quad; \quad \theta_{t+1} = \theta_t - \alpha v_t$$
- Accumulates velocity, dampens oscillations
- Passes through local minima more easily
- $\beta$ typically 0.9

### 3. Adagrad
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{G_t + \epsilon}} \cdot g_t$$
where $G_t = \sum_{i=1}^{t} g_i^2$ (accumulated squared gradients)
- Per-parameter adaptive learning rate
- Good for sparse features (NLP)
- **Problem**: LR decays to zero (accumulated denominator grows monotonically)

### 4. RMSprop
$$G_t = \beta G_{t-1} + (1-\beta) g_t^2 \quad; \quad \theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{G_t + \epsilon}} g_t$$
- Fixes Adagrad's decaying LR with exponential moving average
- Default $\beta = 0.99$
- Good for RNNs

### 5. Adam (Adaptive Moment Estimation)
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad (\text{1st moment: mean})$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad (\text{2nd moment: variance})$$
$$\hat{m}_t = m_t / (1-\beta_1^t), \quad \hat{v}_t = v_t / (1-\beta_2^t) \quad (\text{bias correction})$$
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$
- Combines momentum + RMSprop
- Defaults: $\beta_1=0.9, \beta_2=0.999, \epsilon=10^{-8}$
- **Most popular** optimizer for deep learning

### 6. AdamW (Adam with Decoupled Weight Decay)
- Standard Adam applies L2 regularization in the gradient step — AdamW decouples it
- Better generalization than Adam
- **Default for Transformers** and modern architectures

### Comparison
| Optimizer | Adaptive LR | Momentum | Best For |
|-----------|-------------|----------|----------|
| SGD | No | No | Final perf (with tuning) |
| SGD+Momentum | No | Yes | CNNs, ResNets |
| Adagrad | Yes | No | Sparse features |
| RMSprop | Yes | No | RNNs |
| Adam | Yes | Yes | General default |
| AdamW | Yes | Yes | Transformers |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 2D optimization landscape: Rosenbrock-like
def f(x, y):
    return (1 - x)**2 + 100*(y - x**2)**2

def grad_f(x, y):
    dx = -2*(1-x) + 200*(y - x**2)*(-2*x)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

def optimize(optimizer_fn, x0, n_steps=500):
    path = [x0.copy()]
    x = x0.copy()
    state = {}
    for t in range(1, n_steps+1):
        g = grad_f(x[0], x[1])
        x = optimizer_fn(x, g, t, state)
        path.append(x.copy())
    return np.array(path)

def sgd(x, g, t, state, lr=0.0001):
    return x - lr * g

def adam(x, g, t, state, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    state.setdefault('m', np.zeros_like(x))
    state.setdefault('v', np.zeros_like(x))
    state['m'] = b1*state['m'] + (1-b1)*g
    state['v'] = b2*state['v'] + (1-b2)*g**2
    m_hat = state['m'] / (1 - b1**t)
    v_hat = state['v'] / (1 - b2**t)
    return x - lr * m_hat / (np.sqrt(v_hat) + eps)

x0 = np.array([-1.0, 1.0])
path_sgd = optimize(sgd, x0)
path_adam = optimize(adam, x0)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
xx, yy = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-1, 3, 200))
zz = f(xx, yy)
ax.contour(xx, yy, np.log(zz+1), levels=30, cmap='viridis', alpha=0.5)
ax.plot(*path_sgd.T, 'r.-', markersize=2, linewidth=1, label='SGD')
ax.plot(*path_adam.T, 'b.-', markersize=2, linewidth=1, label='Adam')
ax.plot(1, 1, 'g*', markersize=15, label='Optimum')
ax.set_title('SGD vs Adam on Rosenbrock Function')
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways
- **Adam** is the safe default — works well without much tuning
- **AdamW** for Transformers (decoupled weight decay matters)
- **SGD + momentum** often gives best final accuracy for CNNs when properly tuned
- Adam's adaptive LR per parameter helps with sparse gradients and saddle points